# Neighborhood Analysis: Cell-Cell Spatial Relationships

Clustering tells you *what* cell types are present. It says nothing about *where* they sit relative to each other and *how* they interact.

Dataset: the same public 10x Genomics Human Lung Cancer FFPE Xenium dataset (~162k cells) as `01_core_workflow.ipynb`.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

import numpy as np
import astrolabe as ab

## Setup

Cell-type labels and a spatial graph are required for everything below. This reproduces the QC → normalize → cluster pipeline from `01_core_workflow.ipynb`.

In [ ]:
mdata = ab.datasets.xenium()

ab.pipeline.filter_xenium(mdata)
ab.pp.normalize_total(mdata)
ab.pp.log1p(mdata)

# Cluster on expression (not space): tl.cluster/tl.umap both default to the
# transcriptomic graph, passed explicitly below for clarity.
ab.pp.highly_variable_genes(mdata, n_top_genes=2000)
ab.tl.pca(mdata)
ab.tl.transcriptional_neighbors(mdata)
ab.tl.cluster(mdata, resolution=0.5, neighbors_key='neighbors')

mdata['xenium'].obs['leiden'].value_counts()

In [ ]:
# Now build the spatial graph -- every function below needs it. Delaunay gives a
# density-adaptive graph (good for variable cell density); pruning long edges at
# 200um avoids linking cells across large segmentation gaps. Coexists independently
# of the transcriptomic graph built above (uns['neighbors'] is left untouched).
ab.tl.spatial_neighbors(mdata, coord_type='delaunay', delaunay_max_dist=200)

## 1. Cell-type adjacency enrichment

A basic test would be: for every pair of clusters (A, B), whether A-cells and B-cells are connected in the spatial graph more often than expected if labels were randomly shuffled across the same tissue. `tl.nhood_enrichment` performs this permutation test, reporting a z-score per pair: strongly positive means "attracted" (enriched adjacency), strongly negative means "segregated" (avoidance).

In [ ]:
ab.tl.nhood_enrichment(mdata, 'leiden')
ab.pl.nhood_enrichment_plot(mdata, 'leiden').show()

Read the strongest signal directly off the z-score matrix rather than assuming which pair it is: the largest positive off-diagonal z-score is the most "attracted" distinct-cluster pair (enriched adjacency, far more graph-neighbor edges between the two than 1,000 label permutations produce by chance); the most negative is the most "segregated" pair. A large positive z between two clusters that turn out to be anatomically adjacent cell types (e.g. a structural/perivascular type next to an endothelial one) is the expected signature of real tissue architecture — check `ab.tl.rank_genes_groups` for what each cluster number actually is before drawing that conclusion, rather than assuming an identity from the number alone. Step 7 below picks out this same top pair programmatically from `uns['nhood_enrichment']` for a deeper look.

## 2. Enrichment across distance scales (co-occurrence)

`nhood_enrichment` collapses everything into a single "are they graph-neighbors" yes/no per edge. Two cell types could be strongly enriched at direct-contact range (~10-20 µm) but indistinguishable from random at 200 µm, or vice versa, with a broad gradient and no sharp boundary. `tl.co_occurrence` refines the same question across a sweep of distance thresholds, computing the ratio P(type B within d of type A) / P(type B globally) at each one. A ratio above 1 is enrichment, below 1 is avoidance, and the distance at which the curve crosses 1 marks the length scale at which the relationship disappears.

In [ ]:
ab.tl.co_occurrence(mdata, 'leiden', interval=(10, 300, 30))
ab.pl.co_occurrence_plot(mdata, 'leiden', source_groups=['0']).show()

The co-occurrence curve for the anchor cluster (`source_groups=['0']`) shows whether same-cluster pairs stay enriched above 1 across the whole distance range tested — a spatially contiguous domain, like a compact tissue nest — or fall back toward 1 quickly, which instead describes a cell type that clusters locally without forming large contiguous regions. Curves for other clusters that dip below 1 at short range and recover at longer range indicate active short-range avoidance of the anchor cluster that weakens with distance. Check the legend for which cluster each curve belongs to, and its actual marker genes (`ab.tl.rank_genes_groups`, as in `01_core_workflow.ipynb`), rather than assuming a biological identity from the cluster number alone — cluster numbering is arbitrary and specific to a given clustering run.

## 3. Structural centrality within the spatial graph

Enrichment and co-occurrence are about pairs. A different question concerns each cell type as a whole: does it sit along the tissue's edges, barely touching anything else, or does it form a densely-connected core that much of the rest of the tissue borders? `tl.centrality_scores` answers this from the spatial graph alone (no expression involved), treating each cluster as a single group (matching squidpy's own `centrality_scores` convention) rather than averaging individual cells' scores: degree centrality (the fraction of *other* cells that touch *at least one* cell of this type), clustering coefficient (the average, over the cluster's own cells, of how often their neighbors are also mutual neighbors), and closeness centrality (how close the group collectively is to every other cell in the tissue).

In [ ]:
ab.tl.centrality_scores(mdata, 'leiden')
ab.pl.centrality_scores_plot(mdata, 'leiden').show()

Closeness centrality is skipped for 154k cells: it is O(n²)-ish per group in a dense graph, so the warning is a scaling limit rather than an error. Because degree and closeness are now group-level (not per-cell averages), a cluster's degree centrality close to 1.0 means nearly every other cell in the tissue touches at least one cell of that type — expected for large, spatially dispersed clusters, and a much weaker signal for small or spatially compact ones, which can only ever reach as many outside neighbors as their own (smaller) boundary allows. Clustering coefficient stays a genuine per-cell average within the group, so it's directly readable as "how often do this cluster's cells' neighbors also neighbor each other" — high values point to cell types that form small, dense local aggregates (e.g. immune cell clusters forming tertiary-lymphoid-structure-like nests) rather than being sparsely interspersed throughout the tissue.

## 4. Expression as a function of distance from a cell type

The previous steps describe *where cells are*, not what they're *doing*. If proximity to a cell type is biologically meaningful, nearby cells of other types should show a measurable expression shift moving away from it, rather than a flat line. `tl.expression_by_distance` computes, for every cell, distance to its nearest cell of a chosen anchor cluster, bins those distances, and reports per-bin mean expression ± SE for chosen genes. The anchor here is cluster `'0'`, tracking `EPCAM` (epithelial cell adhesion molecule) and `PTPRC` (CD45, pan-immune marker) to test whether either gradient moves with distance from it.

Which cluster number corresponds to which cell type is specific to a given clustering run — check `ab.tl.rank_genes_groups` (as in `01_core_workflow.ipynb`) before assuming a cluster's biological identity from its number alone, especially in a notebook like this one that reruns its own independent clustering with different HVG settings than other notebooks.

In [ ]:
print(mdata['xenium'].obs['leiden'].cat.categories.tolist())

anchor_cluster = '0'
ab.tl.expression_by_distance(mdata, 'leiden', anchor_cluster, genes=['EPCAM', 'PTPRC'])
ab.pl.expression_by_distance_plot(mdata, 'EPCAM', 'PTPRC').show()

Read the two curves directly rather than assuming which one "belongs" to the anchor cluster: whichever gene is elevated right at the anchor (bin 0 is dominated by the anchor cells' own signal, since a cell sits at distance 0 from itself) is the one enriched in the anchor cluster, and how sharply it falls off moving outward describes how quickly that identity fades with distance. The other gene's opposite trend describes the compartment the anchor cluster is embedded in or borders. `max_dist` (printed above) is the actual outer edge of the plotted range — cells beyond it are excluded from the analysis entirely, not folded into the last bin. This distance-resolved anti-correlation, when both genes trend in opposite directions, is the same kind of relationship the bivariate Moran's I in `02_spatial_statistics.ipynb` captures as a single global number.

## 5. A candidate mechanism: ligand-receptor signaling at the interface

With spatially enriched cluster pairs identified (steps 1–2), `tl.ligrec` tests ligand-receptor signaling directly: for every ligand-receptor pair and every directed (sender, receiver) cluster pair, it scores the mean ligand × receptor product over actual spatial-graph edges and tests it against a permuted null. Restricting to real spatial edges, rather than bulk cluster-level co-expression, makes this a *spatial* ligand-receptor test.

`EDN1`/`EDNRB` (endothelin signaling, vascular/stromal crosstalk) is present in this lung cancer panel after filtering.

In [ ]:
import pandas as pd

lr_pairs = pd.DataFrame({'ligand': ['EDN1'], 'receptor': ['EDNRB']})
ab.tl.ligrec(mdata, 'leiden', lr_pairs)
ab.pl.ligrec_plot(mdata).show()

The log line above reports how many of the cell-type-pair combinations are significant at p < 0.05 for the EDN1→EDNRB pair — a nonzero count here means active endothelin signaling is detectable across at least some cell-type interfaces, restricted to pairs that are actually adjacent in the spatial graph (not bulk co-expression). In lung cancer, endothelin signaling contributes to vascular remodeling and tumor-stroma crosstalk, so finding it active is consistent with a vascularized tumor microenvironment. The `ligrec_plot` dot-plot below shows which (sender, receiver) pairs actually drive the signal — check whether pairs involving vascular/endothelial cell types dominate, rather than assuming it from a cluster number.

## 6. Zooming out: regional cell-type composition

The preceding steps characterize cell-to-cell relationships. At a coarser scale, the question is whether tissue regions have a distinct overall cell-type composition. `tl.composition` answers this, but requires a "region" label distinct from the cell-type label itself. KMeans on µm spatial coordinates carves the tissue into compact spatial zones here, without using expression.

In [ ]:
from sklearn.cluster import KMeans

xen = mdata['xenium']
coords = xen.obsm['spatial']
xen.obs['region'] = pd.Categorical(
    KMeans(n_clusters=4, random_state=0, n_init=10).fit_predict(coords).astype(str)
)

comp = ab.tl.composition(mdata, groupby='leiden', region_by='region')
comp

Reading across the printed table for the region/cluster combination that deviates most from the others: region 2 stands out here, with roughly 40% of its cells in cluster 4 versus roughly 1–9% for that same cluster in the other three regions — a clear several-fold excess, and larger than any other single cluster/region combination in the table. That kind of standout fraction, not a small relative shift, is what marks a region as compositionally distinct rather than just noisy. Regions 0, 1, 3 are comparatively balanced across clusters, without any one cluster dominating a region the way cluster 4 dominates region 2 here. As above, confirm what cluster 4 actually is (`ab.tl.rank_genes_groups`) before drawing a biological conclusion from its number.

In [ ]:
# Sanity check: do the KMeans regions look spatially coherent, and does composition vary across them?
(ab.pl.points(mdata, color='region') | ab.pl.points(mdata, color='leiden')).show()

## 7. Transcriptome-level evidence: interaction-changed genes

With candidate adjacencies (steps 1–2) and a candidate mechanism (step 5), the final step tests whether physical contact actually changes gene expression. `tl.interaction_changed_genes` compares, gene by gene, cells of type A that are touching cells of type B against A-cells that are not, in a single call across all cluster pairs. The result is then filtered to the most enriched distinct-cluster pair from step 1's z-score matrix.

In [ ]:
# Pick the most enriched *distinct-cluster* pair from step 1's z-score matrix
# (off-diagonal, since same-cluster self-adjacency isn't a "contact" question).
nhood = mdata['xenium'].uns['nhood_enrichment']
z, groups = nhood['zscore'].copy(), nhood['groups']
np.fill_diagonal(z, -np.inf)
i, j = np.unravel_index(np.argmax(z), z.shape)
receiver, neighbor = groups[i], groups[j]
print(f"Most-enriched distinct pair: receiver={receiver!r}, neighbor={neighbor!r} (z={z[i, j]:.2f})")

icg = ab.tl.interaction_changed_genes(mdata, groupby='leiden')
pair_hits = icg[(icg['cell_type'] == receiver) & (icg['neighbor_type'] == neighbor)]
pair_hits.sort_values('pval_adj').head(10)

Read the top rows directly: a gene with a large positive `log2fc` and small `pval_adj` is strongly upregulated in the receiver cluster specifically when it's in contact with the neighbor cluster identified above, versus receiver cells that aren't in contact with it. If those top genes turn out to be markers of the *neighbor* cluster's own cell type rather than genes normally associated with the receiver, that's consistent with the receiver picking up a transcriptional signature at the point of contact — though it's also exactly the pattern segmentation spillover produces (transcripts from an adjacent cell mis-assigned into the receiver's polygon), so a real biological contact effect and a segmentation artifact can look identical from this table alone. `seg.create_compartment_layers` + this same function's `layer=` parameter (undocumented but functional) let you re-run this test restricted to a cell's nuclear/interior transcripts only, which a spillover effect should not survive but a true paracrine signaling effect would.

## Summary

This notebook worked through cell-type adjacency at increasing resolution: whether clusters touch more than chance (z-scores), at what length scale (co-occurrence), which cluster is structurally central (centrality), whether expression shifts with distance (expression-by-distance), what ligand-receptor pair could mediate it (ligrec), the regional cell-type composition (composition), and whether contact measurably changes the transcriptome (interaction-changed genes). Each step narrowed from a spatial pattern to a testable biological claim.

For trajectory or gene-module follow-up on a ligand-receptor hit from step 5, see `06_trajectory_and_gene_modules.ipynb`.

## 8. Cell-type spatial colocation quotients (CLQ)

Sections 1–2 asked whether cell types are enriched as graph-neighbors (`nhood_enrichment`) or within a swept distance (`co_occurrence`). The colocation quotient asks a related but distinctly *directional* question: given a cell of type A, is a nearby cell of type B over- or under-represented among A's k nearest spatial neighbors, relative to B's overall abundance in the tissue?

```
CLQ(A→B) = (C(A→B) / (N_A × k)) / (N_B / (N − 1))
```

where `C(A→B)` counts type-B points among the k nearest neighbors of every type-A point. Values:

- **CLQ > 1** — B is over-represented near A (A "attracts" B)
- **CLQ = 1** — no spatial association beyond B's overall abundance
- **CLQ < 1** — B is under-represented near A (A "avoids" B)

Unlike `nhood_enrichment`'s symmetric z-score, `CLQ(A→B)` need not equal `CLQ(B→A)` — a real, asymmetric spatial relationship (Leslie & Kronenfeld 2011). `groupby` takes the same categorical `obs` column used throughout this notebook — no separate spatial graph is needed, since the k-nearest-neighbor structure is derived internally from `k`.

In [ ]:
# CLQ(A->B): is cell type B over/under-represented among cell type A's nearest
# spatial neighbors, relative to B's overall abundance in the tissue?
clq = ab.tl.colocation_quotient(mdata, groupby='leiden', n_perms=200, seed=0)
print(f"Computed CLQ for {len(clq)} ordered cell-type pairs")
clq.query("category_a != category_b").sort_values('clq', ascending=False).head(10)

`CLQ(A→B) > 1` means type B is over-represented among type A's nearest neighbors relative to B's overall abundance — a genuinely spatial, directional statement, unlike `nhood_enrichment`'s symmetric per-edge z-score or `co_occurrence`'s undirected distance-threshold ratio. The strongest rows here should echo the same anatomically adjacent cell-type pairs `nhood_enrichment` found in step 1, since all three methods are measuring versions of the same underlying spatial organization — agreement across independent statistics is a sturdier conclusion than any one of them alone. Same-type pairs (`category_a == category_b`, excluded above) are also present in the full `clq` table and describe how tightly each type clusters with itself, distinct from cross-type attraction.

In [ ]:
# Cell-type pairs whose CLQ is furthest below 1 -- types that actively avoid
# being each other's nearest neighbors, beyond what their abundances alone predict.
clq.query("category_a != category_b").sort_values('clq').head(10)

CLQ well below 1 identifies cell-type pairs that spatially avoid each other beyond what chance proximity would predict — the geometric complement of the enrichment view above. A pair recurring here and at the strongly-negative end of step 1's `nhood_enrichment` z-scores is a consistent segregation signal across two independent spatial statistics, not an artifact of either one on its own.

In [ ]:
# CLQ is asymmetric by construction: "how often is B near A" need not equal
# "how often is A near B" -- e.g. a rare cell type can be tightly surrounded by
# a common one (high CLQ(rare->common)) while the common type barely notices
# the rare one among its own neighbors (low CLQ(common->rare)).
pivot = clq.pivot(index='category_a', columns='category_b', values='clq')
asymmetry = (pivot - pivot.T).abs()
np.fill_diagonal(asymmetry.values, -1)  # exclude the meaningless self-diagonal
i, j = np.unravel_index(np.argmax(asymmetry.to_numpy()), asymmetry.shape)
a, b = asymmetry.index[i], asymmetry.columns[j]
print(f"Most asymmetric pair: CLQ({a}->{b})={pivot.loc[a, b]:.2f}  vs  CLQ({b}->{a})={pivot.loc[b, a]:.2f}")

pivot.loc[[a, b], [a, b]]

The most asymmetric pair above illustrates why direction matters: a small, tightly localized population can be almost entirely surrounded by a much more abundant neighbor type (a high CLQ in that direction) while barely registering as a neighbor from the abundant type's own perspective — its own nearest neighbors are overwhelmingly its own type or other common types, so the rare type doesn't show up disproportionately in the reverse direction. This directional structure is invisible to `nhood_enrichment`'s symmetric z-score and to `co_occurrence`'s undirected ratio; CLQ is the tool in this notebook built specifically to surface it.